# 01 - Data Analysis: Bootstrap Pipeline Benchmark

This notebook processes the raw timing log produced by `src/run_experiment.py`, aggregates it and exports the figures used in the README.

**System under test:** bootstrap confidence intervals for OLS coefficients (the core of the Task 2 pipeline), run on synthetic data.
**Methods compared:** `baseline` (one fit per resample in a Python loop) and `proposed` (all resamples solved in one vectorized batch).

Structure of the notebook:

1. Data ingestion and warm-up filtering
2. Statistical aggregation
3. Visualization and export
4. Conclusions

Reusable logic lives in `src/process_data.py` and `src/visualize.py`; the notebook calls these modules instead of duplicating code.

In [ ]:
import sys
from pathlib import Path

import numpy as np
import pandas as pd
from IPython.display import Image, display

# Find the repository root whether the notebook is opened from /notebooks or from the root
ROOT = Path.cwd()
if ROOT.name == "notebooks":
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT / "src"))

import process_data
import visualize

SEED = 42
np.random.seed(SEED)  # fixed seed for sampling and bootstrapping

RAW_PATH = ROOT / "data" / "raw" / "test_run.jsonl"
SUMMARY_PATH = ROOT / "data" / "processed" / "metrics_summary.csv"
FIG_DIR = ROOT / "reports" / "figures"

pd.set_option("display.width", 160)
print("Repository root:", ROOT.name)
print("numpy", np.__version__, "| pandas", pd.__version__)

## 1. Data ingestion and warm-up filtering

The raw log contains one JSON object per run. For every method and load level the first 5 runs are **warm-up** runs and the next 50 are **evaluation** runs. Warm-up runs capture cold-start effects (memory allocation, cache and thread start-up), so they are discarded before any statistics are computed.

In [ ]:
raw = process_data.load_raw(RAW_PATH)
print(f"{len(raw)} runs loaded from {RAW_PATH.name}")
display(raw.head())
display(raw.groupby(["method", "phase"]).size().unstack())

**Cold-start check.** The table below compares the very first run of each (method, load) combination with the median of its evaluation runs. A ratio above 1 shows a cold-start spike; a ratio close to 1 means the effect is small for that case. Warm-up runs are discarded in every case as a standard precaution.

In [ ]:
first_run = raw.sort_values("run_id").groupby(["method", "n_respondents"]).first()["latency_ms"]
eval_median = (raw[raw["phase"] == "evaluation"]
               .groupby(["method", "n_respondents"])["latency_ms"].median())
cold = pd.DataFrame({"first_run_ms": first_run, "evaluation_median_ms": eval_median})
cold["first / median"] = cold["first_run_ms"] / cold["evaluation_median_ms"]
display(cold.round(3))

In [ ]:
evaluation = raw[(raw["phase"] == "evaluation") & (~raw["error"])]
n_eval_all = int((raw["phase"] == "evaluation").sum())
print(f"Evaluation runs kept: {len(evaluation)} of {n_eval_all}")
print("Failed runs in the evaluation phase:", int(raw[raw["phase"] == "evaluation"]["error"].sum()))

## 2. Statistical aggregation

For every (method, load) pair the evaluation runs are summarized by the median, mean, standard deviation, quartiles (p25, p75), tail percentiles (p95, p99), a bootstrap 95% confidence interval of the median (2000 resamples, seed 42) and the error rate.

In [ ]:
summary = process_data.summarize(raw, np.random.default_rng(SEED))
cols = ["method", "n_respondents", "n_runs", "median_ms", "mean_ms", "sd_ms",
        "p25_ms", "p75_ms", "p95_ms", "p99_ms", "ci_low_ms", "ci_high_ms",
        "error_rate", "peak_mem_mb_median"]
display(summary[cols].round(3))

In [ ]:
# Consistency check: the notebook must reproduce the file written by src/process_data.py
saved = pd.read_csv(SUMMARY_PATH)
pd.testing.assert_frame_equal(
    saved[cols].reset_index(drop=True),
    summary[cols].round(4).reset_index(drop=True),
    check_dtype=False, atol=1e-3,
)
print("OK: the aggregation matches data/processed/metrics_summary.csv")

**Summary table for the README.** Latency is reported as the median with the interquartile range (p25-p75). The speed-up is the ratio of the baseline median to the proposed median.

In [ ]:
pivot = summary.pivot(index="n_respondents", columns="method")


def fmt(method):
    return (pivot[("median_ms", method)].map("{:.2f}".format)
            + " (" + pivot[("p25_ms", method)].map("{:.2f}".format)
            + "-" + pivot[("p75_ms", method)].map("{:.2f}".format) + ")")


readme_table = pd.DataFrame({
    "Baseline median (p25-p75), ms": fmt("baseline"),
    "Proposed median (p25-p75), ms": fmt("proposed"),
    "Speed-up (x)": (pivot[("median_ms", "baseline")] / pivot[("median_ms", "proposed")]).round(2),
    "Baseline p95, ms": pivot[("p95_ms", "baseline")].round(2),
    "Proposed p95, ms": pivot[("p95_ms", "proposed")].round(2),
})
readme_table.index.name = "Respondents"
display(readme_table)


def to_markdown(df):
    header = "| " + " | ".join([df.index.name] + list(df.columns)) + " |"
    sep = "|" + "---|" * (len(df.columns) + 1)
    rows = ["| " + " | ".join([str(i)] + [str(v) for v in r]) + " |"
            for i, r in zip(df.index, df.values)]
    return "\n".join([header, sep] + rows)


print(to_markdown(readme_table))

In [ ]:
mem = summary.pivot(index="n_respondents", columns="method", values="peak_mem_mb_median").round(3)
mem.columns.name = None
print("Median peak memory per run, MB:")
display(mem)

## 3. Visualization and export

The figures follow three rules: labelled axes with units (and log scales because the values span orders of magnitude), uncertainty shown as 95% bootstrap confidence intervals, and a colorblind-safe palette with different markers and line styles per method. Every figure is saved as a vector `.pdf` and a 300 DPI `.png` in `reports/figures/`.

In [ ]:
visualize.apply_style()
visualize.plot_latency_vs_load(summary, FIG_DIR)
visualize.plot_distribution_boxplot(raw, FIG_DIR)
for f in sorted(FIG_DIR.glob("fig*")):
    print(f.relative_to(ROOT), f"{f.stat().st_size / 1024:.0f} KB")

In [ ]:
display(Image(filename=str(FIG_DIR / "fig1_latency_vs_load.png"), width=520))
display(Image(filename=str(FIG_DIR / "fig2_distribution_boxplot.png"), width=520))

## 4. Conclusions

The statements below are computed from the data, so they always match the tables above.

In [ ]:
sp = readme_table["Speed-up (x)"]
mem_ratio = mem["proposed"] / mem["baseline"]
print(f"1. The proposed method is {sp.min():.1f}x to {sp.max():.1f}x faster than the baseline "
      f"(median latency) across all loads.")
print(f"2. Maximum error rate over all methods and loads: {summary['error_rate'].max():.1%}.")
print(f"3. The very first run is at most {cold['first / median'].max():.2f}x the evaluation median; "
      f"the cold-start effect is small here, but warm-up runs are still discarded as a precaution.")
print(f"4. The proposed method uses {mem_ratio.min():.0f}x to {mem_ratio.max():.0f}x more peak memory "
      f"than the baseline: speed is bought with memory.")

**Limitations.** All data are synthetic and were measured on a single machine, so absolute times depend on the hardware. The confidence intervals of the median are narrow because timing noise is small compared with the median. The benchmark describes the performance of the analysis pipeline, not the conclusions of the survey study.